In [11]:
# ── MongoDB Data Manager ──
# Run this cell to list all pairs with stats, then uncomment a delete block below.
# Cleans: candles, candle_features, market_trades, symbol_metadata

import os
from datetime import datetime, timezone
from pymongo import MongoClient

MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.environ.get("MONGO_DATABASE", "quants_lab")

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

COLLECTIONS_WITH_INTERVAL = ["candles", "candle_features"]
COLLECTIONS_PAIR_ONLY = ["market_trades", "symbol_metadata"]
ALL_COLLECTIONS = COLLECTIONS_WITH_INTERVAL + COLLECTIONS_PAIR_ONLY

now_ts = datetime.now(timezone.utc).timestamp()

# ── Discover all connector/pair/interval combos from candles ──
pipeline = [
    {"$group": {
        "_id": {"connector": "$connector", "trading_pair": "$trading_pair", "interval": "$interval"},
        "count": {"$sum": 1},
        "first_ts": {"$min": "$timestamp"},
        "last_ts": {"$max": "$timestamp"},
    }},
    {"$sort": {"_id.connector": 1, "_id.trading_pair": 1, "_id.interval": 1}},
]

combos = list(db["candles"].aggregate(pipeline))

if not combos:
    print("No candle data found in MongoDB.")
else:
    print(f"{'='*100}")
    print(f"  Found {len(combos)} connector/pair/interval combinations in '{DB_NAME}.candles'")
    print(f"{'='*100}\n")
    print(f"  {'#':>4}  {'Connector':<12} {'Trading Pair':<18} {'Intv':<6} "
          f"{'Candles':>10} {'Total Days':>11} {'Stale Days':>11} {'First Candle':<20} {'Last Candle':<20}")
    print(f"  {'─'*4}  {'─'*12} {'─'*18} {'─'*6} {'─'*10} {'─'*11} {'─'*11} {'─'*20} {'─'*20}")

    for i, doc in enumerate(combos):
        connector = doc["_id"]["connector"]
        pair = doc["_id"]["trading_pair"]
        interval = doc["_id"]["interval"]
        count = doc["count"]
        first_ts = doc["first_ts"]
        last_ts = doc["last_ts"]
        total_days = (last_ts - first_ts) / 86400
        stale_days = (now_ts - last_ts) / 86400
        first_dt = datetime.fromtimestamp(first_ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")
        last_dt = datetime.fromtimestamp(last_ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")

        stale_flag = " ⚠️" if stale_days > 7 else ""
        print(f"  [{i:>2}]  {connector:<12} {pair:<18} {interval:<6} "
              f"{count:>10,} {total_days:>10.1f}d {stale_days:>10.1f}d{stale_flag} {first_dt:<20} {last_dt:<20}")

    # Also show doc counts in related collections
    print(f"\n  Related collection sizes:")
    for coll_name in ALL_COLLECTIONS:
        cnt = db[coll_name].estimated_document_count()
        print(f"    {coll_name:<20} {cnt:>12,} docs")


  Found 331 connector/pair/interval combinations in 'quants_lab.candles'

     #  Connector    Trading Pair       Intv      Candles  Total Days  Stale Days First Candle         Last Candle         
  ────  ──────────── ────────────────── ────── ────────── ─────────── ─────────── ──────────────────── ────────────────────
  [ 0]  coinbase     BTC-USD            15m        19,042      198.3d        0.2d 2025-09-09 07:45     2026-03-26 16:00    
  [ 1]  coinbase     BTC-USD            1d            198      197.0d        1.9d 2025-09-09 00:00     2026-03-25 00:00    
  [ 2]  coinbase     BTC-USD            1h          4,761      198.3d        0.3d 2025-09-09 07:00     2026-03-26 15:00    
  [ 3]  coinbase     BTC-USD            1m        285,645      198.4d        0.2d 2025-09-09 07:31     2026-03-26 16:15    
  [ 4]  coinbase     BTC-USD            5m         57,126      198.4d        0.2d 2025-09-09 07:45     2026-03-26 16:10    
  [ 5]  coinbase     ETH-USD            15m        19,042 

In [12]:
# ═══════════════════════════════════════════════════════════════════
# UNCOMMENT ONE of the blocks below, then re-run this cell.
# All deletes clean candles + candle_features + market_trades + symbol_metadata.
# ═══════════════════════════════════════════════════════════════════

# ── Delete a single pair (all intervals) by index from the list above ──
idx = 223   # <-- change to the [#] you want to delete
c = combos[idx]["_id"]
target_connector, target_pair = c["connector"], c["trading_pair"]
confirm = input(f"Delete ALL data for {target_connector} / {target_pair}? Type YES: ")
if confirm == "YES":
    for coll_name in ALL_COLLECTIONS:
        filt = {"connector": target_connector, "trading_pair": target_pair}
        result = db[coll_name].delete_many(filt)
        print(f"  {coll_name:<20} deleted {result.deleted_count:,} docs")
    print(f"\nDone — removed {target_connector} / {target_pair}")
else:
    print("Aborted.")

# ── Delete a specific connector/pair/interval combo by index ──
# idx = 0   # <-- change to the [#] you want to delete
# c = combos[idx]["_id"]
# target_connector, target_pair, target_interval = c["connector"], c["trading_pair"], c["interval"]
# confirm = input(f"Delete {target_connector} / {target_pair} / {target_interval}? Type YES: ")
# if confirm == "YES":
#     for coll_name in COLLECTIONS_WITH_INTERVAL:
#         filt = {"connector": target_connector, "trading_pair": target_pair, "interval": target_interval}
#         result = db[coll_name].delete_many(filt)
#         print(f"  {coll_name:<20} deleted {result.deleted_count:,} docs")
#     # market_trades and symbol_metadata don't have interval, skip unless you want full pair removal
#     print(f"\nDone — removed {target_connector} / {target_pair} / {target_interval} from candles + candle_features")
#     print(f"  (market_trades and symbol_metadata untouched — use 'Delete single pair' to clean those too)")
# else:
#     print("Aborted.")

# ── Delete multiple pairs by index ──
# indices = [0, 3, 5]   # <-- change to the [#] indices you want to delete
# targets = [(combos[i]["_id"]["connector"], combos[i]["_id"]["trading_pair"]) for i in indices]
# print("Will delete ALL data for:")
# for conn, pair in targets:
#     print(f"  {conn} / {pair}")
# confirm = input("Type YES to proceed: ")
# if confirm == "YES":
#     for conn, pair in targets:
#         total = 0
#         for coll_name in ALL_COLLECTIONS:
#             result = db[coll_name].delete_many({"connector": conn, "trading_pair": pair})
#             total += result.deleted_count
#         print(f"  {conn} / {pair}: {total:,} docs removed across all collections")
#     print("\nDone.")
# else:
#     print("Aborted.")

# ── Delete an entire connector (all pairs, all intervals) ──
# TARGET_CONNECTOR = "mexc"   # <-- change this
# confirm = input(f"Delete ALL data for connector '{TARGET_CONNECTOR}'? Type YES: ")
# if confirm == "YES":
#     for coll_name in ALL_COLLECTIONS:
#         result = db[coll_name].delete_many({"connector": TARGET_CONNECTOR})
#         print(f"  {coll_name:<20} deleted {result.deleted_count:,} docs")
#     print(f"\nDone — removed all {TARGET_CONNECTOR} data.")
# else:
#     print("Aborted.")

# ── Delete ALL stale pairs (last candle > N days old) ──
# MAX_STALE = 30   # <-- pairs with last candle older than this many days
# stale_pairs = set()
# for doc in combos:
#     stale_d = (now_ts - doc["last_ts"]) / 86400
#     if stale_d > MAX_STALE:
#         stale_pairs.add((doc["_id"]["connector"], doc["_id"]["trading_pair"]))
# if not stale_pairs:
#     print(f"No pairs stale beyond {MAX_STALE} days.")
# else:
#     print(f"Will delete {len(stale_pairs)} pair(s) stale > {MAX_STALE}d:")
#     for conn, pair in sorted(stale_pairs):
#         print(f"  {conn} / {pair}")
#     confirm = input("Type YES to proceed: ")
#     if confirm == "YES":
#         for conn, pair in sorted(stale_pairs):
#             total = 0
#             for coll_name in ALL_COLLECTIONS:
#                 result = db[coll_name].delete_many({"connector": conn, "trading_pair": pair})
#                 total += result.deleted_count
#             print(f"  {conn} / {pair}: {total:,} docs removed")
#         print("\nDone.")
#     else:
#         print("Aborted.")

# ── Nuclear: delete EVERYTHING in all 4 collections ──
# confirm = input(f"DELETE ALL DATA in {DB_NAME}? Type DELETE EVERYTHING: ")
# if confirm == "DELETE EVERYTHING":
#     for coll_name in ALL_COLLECTIONS:
#         result = db[coll_name].delete_many({})
#         print(f"  {coll_name:<20} deleted {result.deleted_count:,} docs")
#     print("\nAll data deleted.")
# else:
#     print("Aborted.")

Delete ALL data for nonkyc / BNB-USDT? Type YES:  YES


  candles              deleted 76,287 docs
  candle_features      deleted 0 docs
  market_trades        deleted 0 docs
  symbol_metadata      deleted 0 docs

Done — removed nonkyc / BNB-USDT
